# Notebook da Camada Gold — `Silver_to_Gold`

Tarefa `to_Gold` do Job — Projeto CineData Analytics (Atividade 3). Executa depois de `to_Silver`.

Este notebook constrói a modelagem dimensional (*star schema*) para consumo do time de BI e, ao final, responde às perguntas de negócio do desafio de analytics. O modelo tem:
- **cinco dimensões:** `dim_movies`, `dim_genres`, `dim_people`, `dim_companies` e `dim_reviews`;
- **três tabelas ponte**, para os atributos multivalorados: `bridge_movie_genre`, `bridge_movie_person` e `bridge_movie_company`;
- **a tabela fato** `fact_movies_performance`, que centraliza as métricas financeiras e de engajamento.

## Mudanças na Atividade 3
- **Modelo inteiro reconstruído a cada execução:** todas as tabelas Gold são gravadas em `overwrite` na mesma tarefa, então dimensões, pontes e fato ficam sempre consistentes entre si.
- **Chaves substitutas idênticas entre execuções:** cada SK é gerada com `row_number()` ordenado pela chave natural, que é única em cada dimensão. Os mesmos dados geram sempre as mesmas chaves.
- **SKs sempre lidas da tabela gravada:** pontes, fato e `dim_reviews` obtêm as SKs por join com as dimensões lidas de volta com `spark.table`, depois de gravadas. O Spark recalcula um DataFrame a cada ação; reutilizar o DataFrame que gerou as chaves poderia recalcular as SKs, e o join usaria valores diferentes dos persistidos. Lendo a tabela, todas usam exatamente as chaves gravadas.
- **Partição única explícita nas janelas das SKs:** uma numeração sequencial exige ordenar todas as linhas da dimensão juntas, então as janelas usam `Window.partitionBy(F.lit(1))`. O processamento continua numa partição só, que é o que o aviso do Spark apontava; a mudança declara essa escolha de forma explícita. É aceitável pelo tamanho das dimensões. Para volumes muito maiores, a alternativa seria uma SK por hash da chave natural (por exemplo, `xxhash64`, que já retorna BIGINT), sem janela.
- **Validação que interrompe o Job:** o `assert` do grão da fato continua ativo. Se houver filmes duplicados, a tarefa falha e o Job fica marcado como falho, em vez de concluir com dados inconsistentes.
- **Perguntas de negócio na mesma tarefa:** as respostas são recalculadas a cada execução, sempre sobre o modelo recém-construído.
- **Saídas de inspeção comentadas:** os `print` e `display` de inspeção estão comentados. A validação do grão e as respostas das perguntas de negócio continuam ativas.

## Decisões adotadas
- **Chaves substitutas com `row_number()`:** geram um BIGINT sequencial e determinístico, desde que a ordenação seja pela chave natural única. O `monotonically_increasing_id()` gera saltos e pode mudar quando o DataFrame é recalculado, e o `sha2` retorna texto em vez de BIGINT.
- **Ordem de construção:** dimensões → pontes e fato. Pontes e fato obtêm as SKs por join com as dimensões, usando a chave natural.
- **Tabelas ponte:** resolvem as relações muitos-para-muitos (um filme tem vários gêneros, e um gênero aparece em vários filmes) sem duplicar o grão da fato. Na ponte de pessoas, o join usa nome **e** tipo, porque a mesma pessoa pode ter uma SK como Ator e outra como Diretor.
- **dim_people:** o grão é nome + tipo de atuação, e as produtoras ficam em `dim_companies`.
- **dim_reviews:** agrega as avaliações por filme (contagem e média arredondada em 2 casas). Avaliações de filmes inexistentes na `dim_movies` são descartadas pelo inner join, porque uma FK precisa apontar para um registro existente.
- **fact_movies_performance:**
  - Contém só filmes com status `Lançado`.
  - Usa *left join* a partir da dimensão, então um filme sem dados financeiros continua na fato, com métricas nulas.
  - O grão (um registro por filme) é validado com `assert`, que interrompe a execução se houver duplicatas.
- **Tipos finais explícitos:** cada coluna recebe `cast` para o tipo exigido pelo enunciado, independentemente do tipo vindo da Silver.
- **Atenção no consumo:** ao juntar a fato com uma ponte, um filme com N gêneros aparece N vezes. Isso é correto para análises por gênero, mas totais gerais devem ser calculados sobre a fato sozinha.

In [0]:
# Configurações gerais
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

In [0]:
# Listando tabelas do schema silver para auxiliar na criação de tabelas do schema gold
# for t in spark.catalog.listTables(silver_schema):
#     print(t.name)

## gold.dim_movies

In [0]:
# gold.dim_movies (origem: silver.tb_info_filmes)

df_silver_info_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_silver_info_filmes
    .withColumn("sk_movie_id", F.row_number().over(Window.partitionBy(F.lit(1)).orderBy(F.col("id_filme"))))
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)


(df_dim_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_movies"))

dim_movies = spark.table(f"{gold_schema}.dim_movies")
#print(f"{gold_schema}.dim_movies: {dim_movies.count()} linhas | "
#      f"sk únicas: {dim_movies.select('sk_movie_id').distinct().count()}")
##display(dim_movies)

## gold.dim_genres


In [0]:
#gold.dim_genres (origem: silver.tb_generos)

df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

df_dim_generos = (
    df_silver_generos.select("genero").distinct()
    .withColumn("sk_genre_id", F.row_number().over(Window.partitionBy(F.lit(1)).orderBy(F.col("genero"))))
    .select(
        F.col("sk_genre_id").cast("bigint"),
        F.col("genero").cast("string"),
    )
)

(df_dim_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_genres"))

dim_genres = spark.table(f"{gold_schema}.dim_genres")
#print(f"{gold_schema}.dim_genres: {dim_genres.count()} linhas | "
#      f"sk únicas: {dim_genres.select('sk_genre_id').distinct().count()}")
#display(dim_genres)

## gold.dim_people

In [0]:
#gold.dim_people (origem: silver.tb_pessoas_empresas)

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(
        Window.partitionBy(F.lit(1)).orderBy(F.col("tipo_pessoa"), F.col("nome_pessoa"))))
    .select(
        F.col("sk_person_id").cast("bigint"),
        F.col("nome_pessoa").cast("string"),
        F.col("tipo_pessoa").cast("string"),
    )
)

(df_dim_people.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_people"))

dim_people = spark.table(f"{gold_schema}.dim_people")
#print(f"{gold_schema}.dim_people: {dim_people.count()} linhas | "
#      f"sk únicas: {dim_people.select('sk_person_id').distinct().count()}")
#display(dim_people)

##  gold.dim_companies

In [0]:
#gold.dim_companies (origem: silver.tb_pessoas_empresas)

df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_companies = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(Window.partitionBy(F.lit(1)).orderBy(F.col("nome_produtora"))))
    .select(
        F.col("sk_company_id").cast("bigint"),
        F.col("nome_produtora").cast("string"),
    )
)

(df_dim_companies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_companies"))

dim_companies = spark.table(f"{gold_schema}.dim_companies")
#print(f"{gold_schema}.dim_companies: {dim_companies.count()} linhas | "
#      f"sk únicas: {dim_companies.select('sk_company_id').distinct().count()}")
#display(dim_companies)

## gold.dim_reviews

In [0]:
#gold.dim_reviews (origem: silver.tb_avaliacoes_usuarios + gold.dim_movies)

df_silver_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
dim_movies = spark.table(f"{gold_schema}.dim_movies")

# resumo por filme
df_resumo_avaliacoes = (
    df_silver_avaliacoes
    .groupBy(F.col("id_filme").cast("string").alias("id_filme"))
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
)

# busca a SK do filme pela chave natural + gera a SK da review
df_dim_reviews = (
    df_resumo_avaliacoes
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(Window.partitionBy(F.lit(1)).orderBy(F.col("sk_movie_id"))))
    .select(
        F.col("sk_review_id").cast("bigint"),
        F.col("sk_movie_id").cast("bigint"),
        F.col("qtd_avaliacoes_usuarios").cast("int"),
        F.col("nota_media_usuarios").cast("double"),
    )
)

(df_dim_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.dim_reviews"))

dim_reviews = spark.table(f"{gold_schema}.dim_reviews")
#print(f"{gold_schema}.dim_reviews: {dim_reviews.count()} linhas | "
#      f"sk únicas: {dim_reviews.select('sk_review_id').distinct().count()} | "
#      f"filmes únicos: {dim_reviews.select('sk_movie_id').distinct().count()}")
#display(dim_reviews.orderBy(F.desc("nota_media_usuarios")))

## tabelas ponte
#### gold.bridge_movie_genre | gold.bridge_movie_person | gold.bridge_movie_company

In [0]:

# definição dos caminhos das tabelas usadas pras pontes
#tabelas da silver
df_silver_generos = spark.table(f"{silver_schema}.tb_generos")
df_silver_pessoas = spark.table(f"{silver_schema}.tb_pessoas_empresas")
#tabelas da gold (dim)
dim_movies = spark.table(f"{gold_schema}.dim_movies")
dim_genres = spark.table(f"{gold_schema}.dim_genres")
dim_people = spark.table(f"{gold_schema}.dim_people")
dim_companies = spark.table(f"{gold_schema}.dim_companies")


#PONTE FILME-GENERO ============================================================================================================================================
#gold.bridge_movie_genre (origem: silver.tb_generos + dim_movies + dim_genres)


df_bridge_movie_genre = (
    df_silver_generos
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(dim_genres.select("sk_genre_id", "genero"), on="genero", how="inner")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_genre_id").cast("bigint"),
    )
    .distinct()
)

(df_bridge_movie_genre.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.bridge_movie_genre"))

bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
print(f"{gold_schema}.bridge_movie_genre: {bridge_genre.count()} linhas | "
      f"filmes: {bridge_genre.select('sk_movie_id').distinct().count()}")
#display(bridge_genre.limit(10))



#PONTE FILME-PESSOA ============================================================================================================================================
#gold.bridge_movie_person (origem: silver.tb_pessoas_empresas + dim_movies + dim_people)



df_bridge_movie_person = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("nome").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(dim_people.select("sk_person_id", "nome_pessoa", "tipo_pessoa"),
          on=["nome_pessoa", "tipo_pessoa"], how="inner")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_person_id").cast("bigint"),
    )
    .distinct()
)

(df_bridge_movie_person.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.bridge_movie_person"))

bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
# print(f"{gold_schema}.bridge_movie_person: {bridge_person.count()} linhas | "
#       f"filmes: {bridge_person.select('sk_movie_id').distinct().count()}")
#display(bridge_person.limit(10))



#PONTE FILME-PRODUTORA ============================================================================================================================================
#gold.bridge_movie_company (origem: silver.tb_pessoas_empresas + dim_movies + dim_companies)


df_bridge_movie_company = (
    df_silver_pessoas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("nome").alias("nome_produtora"),
    )
    .join(dim_movies.select("sk_movie_id", "id_filme"), on="id_filme", how="inner")
    .join(dim_companies.select("sk_company_id", "nome_produtora"), on="nome_produtora", how="inner")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("sk_company_id").cast("bigint"),
    )
    .distinct()
)

(df_bridge_movie_company.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.bridge_movie_company"))

bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")
print(f"{gold_schema}.bridge_movie_company: {bridge_company.count()} linhas | "
      f"filmes: {bridge_company.select('sk_movie_id').distinct().count()}")
#display(bridge_company.limit(10))


## gold.fact_movies_performance

In [0]:
#gold.fact_movies_performance (origem: dim_movies + silver.tb_financeiro_filmes + silver.tb_metricas_engajamento)

dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_silver_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")

# base: filmes lançados (1 linha por filme)
df_filmes_lancados = (
    dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

# métricas, com a chave natural no mesmo tipo da dimensão
df_financeiro = df_silver_financeiro.withColumn("id_filme", F.col("id_filme").cast("string"))
df_metricas = df_silver_metricas.withColumn("id_filme", F.col("id_filme").cast("string"))

df_fact_performance = (
    df_filmes_lancados
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_metricas, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint"),

        # métricas financeiras
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),

        # métricas de engajamento
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

(df_fact_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{gold_schema}.fact_movies_performance"))

# validação do grão: 1 linha por filme
fact = spark.table(f"{gold_schema}.fact_movies_performance")
qtd_linhas = fact.count()
qtd_filmes = fact.select("sk_movie_id").distinct().count()
print(f"{gold_schema}.fact_movies_performance: {qtd_linhas} linhas | filmes distintos: {qtd_filmes}")
assert qtd_linhas == qtd_filmes, "Grão violado: há filmes duplicados na fato!"

#display(fact.orderBy(F.desc("receita_usd")))